# Train the wiping Diffusion Policies on Colab

Trains one state-based Diffusion Policy per action label on the CSV dataset
(`scripts/export_dataset.py`, uploaded to Hugging Face with `scripts/upload_dataset_hf.py`).
Training needs only `torch`, `numpy`, `pandas`, `huggingface_hub`: no Drake.

**Runtime → Change runtime type → GPU (T4 is enough).** The default 65.7M-parameter U-Net uses < 3 GB of
GPU memory at batch 256. Runs are written to Google Drive and resume from `ckpt_last.pt`, so if the
session disconnects just run the training cell again.

Closed-loop evaluation in Drake is CPU-bound (Colab has 2 vCPUs); copy the run folder back and run
`python -m diffusion_policy.eval` locally instead (last cell shows the Colab way).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# settings
GITHUB_REPO = "https://github.com/YourSaDady/wiping_model.git"   # private repo: https://<token>@github.com/...
HF_DATASET  = "Lo-Fi-gahara/wipe-labels"                            # the repo you uploaded with scripts/upload_dataset_hf.py
K = 500                                                          # controller K_n folder of the dataset
RUNS = "/content/drive/MyDrive/wipe_runs"                        # where runs are written (persistent)HF_MODELS   = "Lo-Fi-gahara/wipe-dp"                           # model repo for the trained weights


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os; os.makedirs(RUNS, exist_ok=True)

In [ ]:
%cd /content
!test -d wiping_model || git clone -q $GITHUB_REPO wiping_model
%cd /content/wiping_model
!git pull -q
!pip -q install huggingface_hub pandas

In [ ]:
# dataset -> data/hf/wipe-labels (only the K folder and the scene lists).
# For a private dataset run `from huggingface_hub import login; login()` first.
from huggingface_hub import snapshot_download
snapshot_download(HF_DATASET, repo_type="dataset", allow_patterns=[f"K{K}/*", "scenes/*", "README.md"],
                  local_dir="data/hf/wipe-labels")
!ls -lh data/hf/wipe-labels/K{K}

## Label (c): x_{t+1} − K⁻¹ f_{t+1}

40 000 steps at batch 256 (≈ 45 epochs of ≈ 228k windows), validation on the 48 held-out demos every
2 000 steps. `--amp` uses fp16 autocast (about 2× faster on a T4); drop it if the loss turns NaN.

In [ ]:
LABEL, SEED = "c", 0
OUT = f"{RUNS}/K{K}_{LABEL}_s{SEED}"
!python -m diffusion_policy.train --data data/hf/wipe-labels --K $K --label $LABEL --seed $SEED --amp --out $OUT

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
def plot(run):
    log = pd.read_csv(f"{run}/log.csv").drop_duplicates("step", keep="last")
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
    ax[0].plot(log.step, log.train_loss, label="train"); ax[0].plot(log.step, log.val_loss, label="val")
    ax[0].set_yscale("log"); ax[0].set_title("denoising loss"); ax[0].legend()
    ax[1].plot(log.step, log.val_a0_err_mm, label="first action"); ax[1].plot(log.step, log.val_exec_err_mm, label="executed 8")
    ax[1].set_title("val chunk error [mm]"); ax[1].legend()
    ax[2].plot(log.step, log.val_exec_err_n_mm); ax[2].set_title("val error along n [mm]")
    for a in ax: a.set_xlabel("step"); a.grid(alpha=0.3)
    plt.suptitle(run.split("/")[-1]); plt.show()
    print(open(f"{run}/best.json").read())
plot(OUT)

## The other labels (same settings)

(a) x_{t+1}, (b) x_{d,t}, (c′) x_{t+1} + K⁻¹(D ẋ_{t+1} − f_{t+1}). On a T4 each run takes ≈ 1–2 h;
run them in separate sessions if needed (each resumes on its own).

In [ ]:
for LABEL in ["b", "cp", "a"]:
    OUT = f"{RUNS}/K{K}_{LABEL}_s{SEED}"
    !python -m diffusion_policy.train --data data/hf/wipe-labels --K $K --label $LABEL --seed $SEED --amp --out $OUT
    plot(OUT)

## Upload the trained weights to Hugging Face

Uploads config + normalizers, `log.csv`, `best.json` and `ema_best.pt` of each run (add `--steps 40000`
for more snapshots). Needs a write token. Others then get them with
`hf download Lo-Fi-gahara/wipe-dp --local-dir runs`.

In [ ]:
from huggingface_hub import login; login()
!python scripts/upload_runs_hf.py --repo $HF_MODELS --runs {RUNS}/K{K}_c_s0

## (Optional) closed-loop check in Drake on Colab

Slow on 2 vCPUs (≈ 1 min per episode); the full validation set (120 scenes) is better run locally:
`python -m diffusion_policy.eval --run runs/K500_c_s0 --split val --tables train --n 10`.

In [ ]:
!pip -q install drake==1.58.0 scipy
!python -m diffusion_policy.eval --run {RUNS}/K{K}_c_s0 --split val --tables tilt00_ke2e4,tilt10_ke1e5 --n 1 --workers 2 --out {RUNS}/K{K}_c_s0/colab_val.jsonl